In [5]:
import Rewrite as rw

In [2]:
base_prompt = """
De leesbaarheidsscore wordt als volgt berekend: 100 - (
        - 4.21
        + (17.28 * woordfrequentie)
        - (1.62 * syntactische afhankelijkheidslengte)
        - (2.54 * inhoudswoorden per bijzin)
        + (16.00 * aandeel concrete zelfstandige naamwoorden)
    )
Ik wil een tekst met een maximale leesbaarheidsscore van 34.
"""

standard_prompt = f"""
Jij gaat mij helpen om een tekst te herschrijven die slecht leesbaar is in een eenvoudigere leesbare tekst.
{base_prompt}
De informatie van de originele tekst moet zoveel als mogelijk behouden blijven in de herschreven, leesbaardere tekst.
Je krijgt een tekst die je moet herschrijven. Je hoeft geen vraag te beantwoorden, alleen de tekst te herschrijven.
"""

roleplay_prompt = f"""
Je speelt de rol van een goede schrijver. Je werkt bij Gebruiker Centraal die overheidsbrieven versimpelt. Je bent goed in het herschrijven van moeilijk leesbare teksten zodat
deze eenvoudiger leesbaar zijn. Je kan altijd teksten met een hoge leesbaarheidsscore omschrijven naar teksten met een leesbaarheidsscore onder de 34.
{base_prompt}
Hoe lager de score, hoe beter. Jij hebt als taak gekregen om de teksten die je krijgt te versimpelen aan de hand van de leesbaarheidsscore.
Het doel is om laaggeletterde mensen de moeilijk leesbare teksten te laten begrijpen. Bij het herschrijven behoud je de inhoud van de tekst en doe je je wijzigingen op zinsniveau.
Je voegt dus geen dingen toe en de structuur van de tekst blijft behouden.
"""

meta_prompt = f"""
{base_prompt}
Het probleem met de volgende tekst is dat deze een leesbaarheidsscore van <doc_score> heeft. De leesbaarheidsscore van de zinnen zijn <sent_scores>.
Herschrijf de tekst zodat het een lagere leesbaarheidsscore heeft. Los het probleem op door het op te splitsen in kleinere stapjes. Geef de nieuwe tekst in het gegeven format
"""

cot_prompt = f"""
Doel: Het doel is om de leesbaarheidsscore van een tekst zo laag mogelijk te maken. Hoe lager de score, hoe beter.
{base_prompt}
De huidige leesbaarheidsscore van de tekst is <doc_score>; het doel is 34 of lager. De huidige scores per zin zijn <sent_scores>.

Methode:
- Denk eerst na voor je de tekst herschrijft. Beredeneer waarom je een aanpassing maakt, verwijs naar de specifieke teksten.
- Bepaal eerst het thema van de tekst. Bepaal op basis daarvan of moeilijke woorden bepalend zijn voor de betekenis van de tekst.
- Herschrijf geen zinnen met een leesbaarheidsscore lager dan 34, behoud deze wel.
- Lange woorden zijn waarschijnlijk moeilijk. Vervang alleen deze door eenvoudigere alternatieven.
- Bijvoeglijknaamwoorden mogen vereenvoudigd of verwijderd worden als ze niet essentieel zijn.
- Soms horen twee woorden in de zin bij elkaar, maar staan ze ver van elkaar af. Die onderlinge afstand noemen we de afhankelijkheidslengte. Herschrijf zodat deze niet langer is dan 8 woorden. 
- Maak van deelzinnen aparte zinnen. Gebruik signaalwoorden om de verbinding te houden met originele hoofdzin.
- Eigennamen en entiteiten moeten behouden blijven.
- Houd de betekenis en de boodschap van de zinnen en de tekst hetzelfde.
- Voeg geen informatie toe.
- Geef de nieuwe tekst in het gegeven format.
"""

meta_prompt_no_metric = f"""
{base_prompt}
Het probleem met de volgende tekst is dat deze een te hoge leesbaarheidsscore heeft.
Herschrijf de tekst zodat het een lagere leesbaarheidsscore heeft. Los het probleem op door het op te splitsen in kleinere stapjes. Geef de nieuwe tekst in het gegeven format
"""

meta_prompt_no_lint_formula = f"""
Het probleem met de volgende tekst is dat deze een te hoge leesbaarheidsscore heeft.
Herschrijf de tekst zodat het een lagere leesbaarheidsscore heeft. Los het probleem op door het op te splitsen in kleinere stapjes. Geef de nieuwe tekst in het gegeven format
"""

In [6]:
rw

<module 'Rewrite' from '/Users/Michiel/Documents/Studie bachelor + master/Master ADS/Thesis/code/Rewrite.py'>

In [7]:
gpt41mini = rw.RE_OpenAi(
    model="gpt-4.1-mini",
    name = "GPT-4.1 mini",
    is_local=False,
    is_open_source=False,
    is_large=True,
    is_dutch=True
)

In [8]:
inp_txt = """De Oudegracht is het sfeervolle hart van de stad.
In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen. 
Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen."""

In [12]:
rw_pipe = rw.LintPipe(cot_prompt, cot_prompt , inp_txt, "sample_txt")
rw_pipe.add_engine(gpt41mini)
rw_pipe.list_engines()

In [13]:
z = rw_pipe.prompt_engines()

In [14]:
z

[{'text': [{'original_sentence': 'De Oudegracht is het sfeervolle hart van de stad.',
    'rewritten_sentences': 'De Oudegracht is het fijne hart van de stad.',
    'changes_in_sentence': [{'change_id': '1',
      'reason_for_change': "Vervangen van 'sfeervolle' door eenvoudiger synoniem 'fijne' om woordfrequentie te verhogen en leesbaarheid te verbeteren.",
      'start_word': 3,
      'end_word': 3,
      'original_text': 'sfeervolle',
      'new_text': 'fijne',
      'rewrite_category': <RewriteCategory.word_frequency: 'verhoog woordfrequentie'>}]},
   {'original_sentence': 'In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.',
    'rewritten_sentences': 'Vroeger, in de middeleeuwen, was het hier erg druk. Er kwamen en gingen veel goederen.',
    'changes_in_sentence': [{'change_id': '2',
      'reason_for_change': 'De lange zin met inhoudswoorden en verbindingswoorden is opgesplitst in twee kortere zinnen om de afhankelijkheidslengte te verlage

In [15]:
rw_pipe.eval_response()

,prompt_id,Org - text,Org - lint,Org - sent,GPT-4.1 mini/gpt-4.1-mini - text,GPT-4.1 mini/gpt-4.1-mini - lint,GPT-4.1 mini/gpt-4.1-mini - sent,GPT-4.1 mini/gpt-4.1-mini - chngs
0,sample_txt,De Oudegracht is het sfeervolle hart van de st...,48,"18, 54, 63",De Oudegracht is het fijne hart van de stad. V...,32,"22, 24, 30, 32",{'text': [{'original_sentence': 'De Oudegracht...
